In [1]:
!pip install timm pycocotools

In [2]:
dataDir = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"

annFile = f"{dataDir}/annotations/instances_train2017.json"
imgDir = f"{dataDir}/train2017"

In [3]:
import os
import shutil
from pycocotools.coco import COCO

shutil.rmtree("/kaggle/working/vww", ignore_errors=True)

output_dir = "/kaggle/working/vww"
os.makedirs(output_dir + "/0", exist_ok=True)
os.makedirs(output_dir + "/1", exist_ok=True)

coco = COCO(annFile)

person_cat_id = coco.getCatIds(catNms=['person'])[0]
img_ids = coco.getImgIds()[:10000]   # 🔥 use 10K images

for img_id in img_ids:
    img_info = coco.loadImgs(img_id)[0]
    ann_ids = coco.getAnnIds(imgIds=img_id, catIds=[person_cat_id])
    anns = coco.loadAnns(ann_ids)

    label = 0
    for ann in anns:
        if ann['area'] / (img_info['width'] * img_info['height']) > 0.1:
            label = 1
            break

    src = os.path.join(imgDir, img_info['file_name'])
    dst = os.path.join(output_dir, str(label), img_info['file_name'])

    if os.path.exists(src):
        shutil.copy(src, dst)

print("Dataset ready ✅")

loading annotations into memory...
Done (t=16.48s)
creating index...
index created!
Dataset ready ✅


In [4]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.RandomAffine(15),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder("/kaggle/working/vww", transform=transform)

print(dataset.class_to_idx)

{'0': 0, '1': 1}


In [5]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [6]:
import timm

model = timm.create_model('mobilevit_xxs', pretrained=True, num_classes=2)

model.safetensors:   0%|          | 0.00/5.14M [00:00<?, ?B/s]

In [7]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [8]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [9]:
epochs = 12

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 76.9731
Epoch 2, Loss: 44.2487
Epoch 3, Loss: 40.7247
Epoch 4, Loss: 29.9747
Epoch 5, Loss: 27.4048
Epoch 6, Loss: 23.4972
Epoch 7, Loss: 23.0177
Epoch 8, Loss: 22.5565
Epoch 9, Loss: 22.8330
Epoch 10, Loss: 21.6369
Epoch 11, Loss: 21.3012
Epoch 12, Loss: 22.7387


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

fp32_acc = correct / total
print("FP32 Accuracy:", fp32_acc)

FP32 Accuracy: 0.9315


In [11]:
model = model.cpu()

In [12]:
import torch.quantization as quant

quantized_model = quant.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

/tmp/ipykernel_22/112709954.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quant.quantize_dynamic(


In [13]:
quantized_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        outputs = quantized_model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

int8_acc = correct / total
print("INT8 Accuracy:", int8_acc)

INT8 Accuracy: 0.938


In [14]:
import os

torch.save(model.state_dict(), "fp32.pth")
torch.save(quantized_model.state_dict(), "int8.pth")

print("FP32 size:", os.path.getsize("fp32.pth")/1e6, "MB")
print("INT8 size:", os.path.getsize("int8.pth")/1e6, "MB")

FP32 size: 3.922458 MB
INT8 size: 2.474472 MB


In [15]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.6 MB/s eta 0:00:00


In [16]:
import torch

dummy = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,   # use FP32 model
    dummy,
    "mobilevit_fp32.onnx",
    opset_version=11
)

W0424 07:38:43.117000 22 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0424 07:38:43.894000 22 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0424 07:38:43.896000 22 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int

[torch.onnx] Obtain model graph for `ByobNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ByobNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Runtim

ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.10.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"linear_36"<FLOAT,[1,2]>
            ),
            initializers=(
                %"stem.conv.weight"<FLOAT,[16,3,3,3]>{Tensor(...)},
                %"stages.0.0.conv1_1x1.conv.weight"<FLOAT,[32,16,1,1]>{Tensor(...)},
                %"stages.0.0.conv2_kxk.conv.weight"<FLOAT,[32,1,3,3]>{Tensor(...)},
                %"stages.0.0.conv3_1x1.conv.weight"<FLOAT,[16,32,1,1]>{Tensor(...)},
                %"stages.1.0.conv1_1x1.conv.weight"<FLOAT,[32,16,1,1]>{Tensor(...)},
                %"stages.1.0.conv2_kxk.conv.weight"<FLOAT,[32,1,3,3]>{Tensor(...)},
                %"stages.1.0.conv3_

In [17]:
# During training, store loss each epoch
train_losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_losses.append(total_loss)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [ ]:
import matplotlib.pyplot as plt

models = ["FP32", "INT8"]
accuracies = [fp32_acc, int8_acc]

plt.bar(models, accuracies)
plt.ylabel("Accuracy")
plt.title("Accuracy Comparison")
plt.ylim(0.9, 1.0)
plt.show()

In [ ]:
sizes = [3.92, 2.47]  # your values in MB

plt.bar(models, sizes)
plt.ylabel("Size (MB)")
plt.title("Model Size Comparison")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
model = model.to(device)
model.eval()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

all_preds = []
all_labels = []

model.eval()
model = model.to(device)   # ✅ important

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)      # move to GPU
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())     # ✅ move back to CPU
        all_labels.extend(labels.cpu().numpy())   # ✅ move back to CPU

cm = confusion_matrix(all_labels, all_preds)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
train_losses = [
    78.1745, 47.4832, 39.7019, 31.3308,
    27.2698, 25.1345, 24.3161, 23.3530,
    23.5823, 23.4577, 23.0523, 22.5431
]

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss vs Epoch")
plt.grid()
plt.show()

In [ ]:
!pip install onnx onnxruntime

In [ ]:
import onnx

model = onnx.load("mobilevit_fp32.onnx")
onnx.checker.check_model(model)

print("Model is valid ✅")

In [ ]:
print(onnx.helper.printable_graph(model.graph))

In [ ]:
!pip install onnxruntime

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession("/kaggle/working/mobilevit_fp32.onnx")
print("ONNX model loaded successfully ✅")

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# pick any image from your dataset
img_path = "/kaggle/working/vww/1/" + os.listdir("/kaggle/working/vww/1")[0]

img = Image.open(img_path).convert("RGB")
plt.imshow(img)
plt.title("Input Image")
plt.axis('off')

In [ ]:
img = img.resize((224, 224))
img = np.array(img).astype(np.float32) / 255.0

# convert to CHW format
img = np.transpose(img, (2, 0, 1))

# add batch dimension
img = np.expand_dims(img, axis=0)

In [ ]:
input_name = session.get_inputs()[0].name
outputs = session.run(None, {input_name: img})

pred = np.argmax(outputs[0])
print("Prediction:", pred)

In [ ]:
label_map = {0: "No Person", 1: "Person"}

print("Predicted Class:", label_map[pred])

In [ ]:
import os

path_0 = "/kaggle/working/vww/0"
path_1 = "/kaggle/working/vww/1"

count_0 = len(os.listdir(path_0))
count_1 = len(os.listdir(path_1))

print("No Person (0):", count_0)
print("Person (1):", count_1)
print("Total Images:", count_0 + count_1)